# Converting BPMN Model to Petri Net with PM4Py

This notebook demonstrates how to convert a BPMN model into a Petri net using PM4Py. Follow the steps below to set up the environment and execute the notebook:

**Set up the environment**:
   - Create a virtual environment named `.venv` in the current workspace.
   - Install the required dependencies listed in `requirements.txt`.


In [ ]:
# verify local pipeline configuration before importing modules
_REQUIRED_CONFIG_FIELDS = (
    "FRONTEND_URL",
    "EMBEDDING_MODEL",
    "LINGUISTIC_PIPELINE",
    "ACTION_WEIGHT",
    "OBJECT_WEIGHT",
    "FULL_LABEL_WEIGHT",
    "LEXICAL_WEIGHT",
    "TOKEN_WEIGHT",
    "MINIMUM_CANDIDATE_MARGIN",
    "CANDIDATE_THRESHOLD",
    "EXPORT_EVALUATION_METRICS",
)

_config_values = {}
try:
    with open(".env", encoding="utf-8") as _env_file:
        for _line in _env_file:
            _line = _line.strip()
            if not _line or _line.startswith("#") or "=" not in _line:
                continue
            _key, _value = _line.split("=", 1)
            _config_values[_key.strip()] = _value.strip().strip('"').strip("'")
except FileNotFoundError as exc:
    raise RuntimeError("Configuration file not set: .env not found") from exc

_missing_config = [
    _field for _field in _REQUIRED_CONFIG_FIELDS
    if not _config_values.get(_field)
]
if _missing_config:
    raise RuntimeError(
        "Configuration file not set: missing values for " + ", ".join(_missing_config)
    )

FRONTEND_URL = _config_values["FRONTEND_URL"]
EMBEDDING_MODEL = _config_values["EMBEDDING_MODEL"]
LINGUISTIC_PIPELINE = _config_values["LINGUISTIC_PIPELINE"]

_export_evaluation_metrics_raw = _config_values["EXPORT_EVALUATION_METRICS"].lower()
if _export_evaluation_metrics_raw not in {"true", "false"}:
    raise RuntimeError(
        "Configuration file not set: EXPORT_EVALUATION_METRICS must be true or false"
    )
EXPORT_EVALUATION_METRICS = _export_evaluation_metrics_raw == "true"

try:
    ACTION_WEIGHT = float(_config_values["ACTION_WEIGHT"])
    OBJECT_WEIGHT = float(_config_values["OBJECT_WEIGHT"])
    FULL_LABEL_WEIGHT = float(_config_values["FULL_LABEL_WEIGHT"])
    LEXICAL_WEIGHT = float(_config_values["LEXICAL_WEIGHT"])
    TOKEN_WEIGHT = float(_config_values["TOKEN_WEIGHT"])
    MINIMUM_CANDIDATE_MARGIN = float(_config_values["MINIMUM_CANDIDATE_MARGIN"])
    CANDIDATE_THRESHOLD = float(_config_values["CANDIDATE_THRESHOLD"])
except ValueError as exc:
    raise RuntimeError("Configuration file not set: numeric configuration values are invalid") from exc

_WEIGHT_TOTAL = ACTION_WEIGHT + OBJECT_WEIGHT + FULL_LABEL_WEIGHT + LEXICAL_WEIGHT + TOKEN_WEIGHT
if abs(_WEIGHT_TOTAL - 1.0) > 1e-6:
    raise RuntimeError(
        f"Configuration file not set: hybrid-score weights must sum to 1.0, got {_WEIGHT_TOTAL:.6f}"
    )

# verify PM4Py installation and import
print ("🛠️ Importing Modules...")
import pm4py
from lxml import etree
import pandas
import re
import numpy 

# keep notebook errors concise (no huge verbose tracebacks)
try:
    get_ipython().run_line_magic("xmode", "Minimal")
except Exception:
    pass

print ("✅ All modules imported!")


🛠️ Importing Modules...
Exception reporting mode: Minimal
✅ All modules imported!


In [ ]:
import warnings
import os

# suppress the ISO8601 datetime parsing warning from PM4Py
warnings.filterwarnings("ignore", message="ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11")

# suppress HuggingFace symlinks warning
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# suppress other common PM4Py warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pm4py")
warnings.filterwarnings("ignore", category=FutureWarning, module="pm4py")

#### Syntax Check for the BPMN Model

- syntax check according to BPMN 2.0 XSD
- be sure to adjust BPMN Model path


In [ ]:
bpmn_path = "dscp-ajuste-direto-simplificado-v2-normalized-adjusted-english_CT.bpmn"
xsd_file_path = './BPMN20/BPMN20.xsd'

# load BPMN XML
with open(bpmn_path, 'rb') as f:
    bpmn_xml = etree.parse(f)

# load XSD schema
with open(xsd_file_path, 'rb') as f:
    schema_root = etree.parse(f)
    schema = etree.XMLSchema(schema_root)

is_valid = schema.validate(bpmn_xml)
print("BPMN syntax valid?", is_valid)

# print validation errors if any
if not is_valid:
    print("\nValidation Errors:")
    validation_error_messages = []
    for error in schema.error_log:
        error_message = f"Line {error.line}: {error.message}"
        validation_error_messages.append(error_message)
        print(error_message)
    raise RuntimeError(
        "BPMN syntax is invalid. " + " | ".join(validation_error_messages)
    )


BPMN syntax valid? True


#### Import BPMN Model


In [4]:
try:
    bpmn_model = pm4py.read_bpmn(bpmn_path)
except Exception as e:
    raise RuntimeError(f"Failed to load BPMN model from: {bpmn_path}") from e

#### Extract Cycle Times from BPMN Activities

In [ ]:
# extract cycle time annotations from BPMN activities.
# this BPMN stores CT values in textAnnotation nodes linked to tasks by association.
cycle_time_pattern = re.compile(
    r"\bCT\s*=\s*([0-9]+(?:[\.,][0-9]+)?)\s*(?:(?:working\s*)?days?|dias?\s+(?:úteis|uteis)|dia\s+(?:útil|util)|dias?)\b",
    re.IGNORECASE,
)

namespaces = {
    "bpmn": "http://www.omg.org/spec/BPMN/20100524/MODEL"
}

# normalise activity labels so ct can be joined with metric keys.
def normalize_activity_key(label):
    if not isinstance(label, str):
        return ""
    label = label.strip().lower()
    label = label.replace('_', ' ').replace('-', ' ')
    label = re.sub(r'[^\w\u00C0-\u017F&]+', ' ', label)
    label = re.sub(r'\s+', ' ', label)
    return label.strip()

# parse ct values written with comma or dot decimal separators.
def parse_cycle_time_days(value):
    days = float(value.replace(",", "."))
    return int(days) if days.is_integer() else days

# all BPMN activities which might have ct annotations
activity_xpath = " | ".join([
    "//bpmn:task",
    "//bpmn:userTask",
    "//bpmn:serviceTask",
    "//bpmn:sendTask",
    "//bpmn:receiveTask",
    "//bpmn:manualTask",
    "//bpmn:businessRuleTask",
    "//bpmn:scriptTask",
    "//bpmn:callActivity",
    "//bpmn:subProcess",
])

bpmn_tree = etree.parse(bpmn_path)
activity_cycle_times = []

activities_by_id = {
    activity.get("id"): activity
    for activity in bpmn_tree.xpath(activity_xpath, namespaces=namespaces)
}

# extrat ct for annotations and store them by annotation id
cycle_times_by_annotation_id = {}
for annotation in bpmn_tree.xpath("//bpmn:textAnnotation", namespaces=namespaces):
    annotation_text = " ".join(annotation.xpath(".//bpmn:text/text()", namespaces=namespaces))
    match = cycle_time_pattern.search(annotation_text)
    if match:
        cycle_times_by_annotation_id[annotation.get("id")] = parse_cycle_time_days(match.group(1))

# connect activities to their ct annotations via associations
activity_ids_to_cycle_times = {}
for association in bpmn_tree.xpath("//bpmn:association", namespaces=namespaces):
    source_ref = association.get("sourceRef")
    target_ref = association.get("targetRef")

    if source_ref in activities_by_id and target_ref in cycle_times_by_annotation_id:
        activity_ids_to_cycle_times[source_ref] = cycle_times_by_annotation_id[target_ref]
    elif target_ref in activities_by_id and source_ref in cycle_times_by_annotation_id:
        activity_ids_to_cycle_times[target_ref] = cycle_times_by_annotation_id[source_ref]

# clean connections
for activity_id, cycle_time_days in activity_ids_to_cycle_times.items():
    activity = activities_by_id[activity_id]
    activity_name = activity.get("name") or activity_id
    clean_activity_name = re.sub(
        r"\s*\(?\bCT\s*=\s*[0-9]+(?:[\.,][0-9]+)?\s*(?:working\s*)?days?\)?",
        "",
        activity_name,
        flags=re.IGNORECASE,
    ).strip() or activity_name

    activity_cycle_times.append({
        "activity": clean_activity_name,
        "normalized_activity": normalize_activity_key(clean_activity_name),
        "cycle_time_days": cycle_time_days,
    })

# create variables
if activity_cycle_times:
    cycle_times_by_activity = {
        item["activity"]: item["cycle_time_days"]
        for item in activity_cycle_times
    }

    cycle_times_activities = {
        item["normalized_activity"]: item["cycle_time_days"]
        for item in activity_cycle_times
    }

    print("✅ Extracted Cycle Times from BPMN Activities")
else:
    cycle_times_by_activity = {}
    cycle_times_activities = {}
    print("ℹ️ No Cycle Times to Extract from BPMN Activities")

#### Convert imported BPMN model to Petri Net


In [ ]:
net, im, fm = pm4py.convert_to_petri_net(bpmn_model)

#### Save Petri Net Image

- save Petri Net image to /img directory
- automatically assigns version number for comparison reasons when soundness is false

In [7]:
# Define directory and base filename
# output_dir = "../img"
# base_filename = "petri-net-v1.png"
# output_path = os.path.join(output_dir, base_filename)

# # Check if the file already exists and generate a unique filename
# counter = 1
# while os.path.exists(output_path):
#     counter += 1
#     output_path = os.path.join(output_dir, f"petri-net-v{counter}.png")

# # Save the Petri net visualization
# pm4py.save_vis_petri_net(net, im, fm, output_path)
# print(f"Petri net visualization saved to '{output_path}'")

#### Validate Petri Net Soundness

- verify if the conversion is faithful to the original model and abides by the Petri Net fundamentals

In [ ]:
from pm4py.algo.analysis.woflan import algorithm as woflan
import io
from contextlib import redirect_stdout, redirect_stderr

buf = io.StringIO()

with redirect_stdout(buf), redirect_stderr(buf):
    woflan_results = woflan.apply(net, im, fm)

# extract soundness result
if isinstance(woflan_results, bool):
    is_sound = woflan_results
else:
    is_sound = woflan_results.get("is_sound", False)

if is_sound:
    print("✅ Petri Net sound")
else:
    print("❌ Petri Net NOT sound")
    raise RuntimeError("BPMN model is not sound. Please upload a sound BPMN model.")

✅ Petri Net sound


### Import Log Data From CSV/XES

- if the log data file is type CSV then it converts it to type XES and saves it
- if the log data file is type XES then just imports it

In [ ]:
from pm4py.objects.log.importer.xes import importer as xes_importer
from pm4py.objects.conversion.log import converter as log_converter
from pm4py.objects.log.exporter.xes import exporter as xes_exporter
from pm4py.objects.log.obj import EventLog, Trace

buf_align = io.StringIO()

# helper: context manager to silence tqdm and IPython display outputs
from contextlib import contextmanager
@contextmanager
def silence_tqdm_and_display():
    try:
        import tqdm
        from IPython import display as ipdisplay
    except Exception:
        # if imports fail, act as a no-op context manager
        yield
        return

    _tqdm_orig = getattr(tqdm, 'tqdm', None)
    _tqdm_nb_orig = getattr(getattr(tqdm, 'notebook', None), 'tqdm', None) if hasattr(tqdm, 'notebook') else None
    _ipydisplay_orig = getattr(ipdisplay, 'display', None)

    def _silent_tqdm(*a, **k):
        # force disable progress bars
        k['disable'] = True
        if _tqdm_orig:
            return _tqdm_orig(*a, **k)
        # fallback: return the iterable if provided
        return a[0] if a else iter([])

    try:
        if _tqdm_orig:
            tqdm.tqdm = _silent_tqdm
        if hasattr(tqdm, 'notebook') and _tqdm_nb_orig:
            tqdm.notebook.tqdm = _silent_tqdm
        ipdisplay.display = lambda *a, **k: None
        yield
    finally:
        try:
            if _tqdm_orig:
                tqdm.tqdm = _tqdm_orig
            if _tqdm_nb_orig and hasattr(tqdm, 'notebook'):
                tqdm.notebook.tqdm = _tqdm_nb_orig
        except Exception:
            pass
        if _ipydisplay_orig:
            ipdisplay.display = _ipydisplay_orig

# === CONFIGURATION ===
file_path = "piabs-abaixo-5000-10000ID-anonymized-english.xes"

if not file_path:
    raise ValueError("file_path is empty. This pipeline expects a file uploaded via backend /run.")

output_folder = "../data/"
converted_xes_filename = "piabs-abaixo-5000-10000ID-lifecycle-converted.xes"

# detect file type based on extension
file_extension = os.path.splitext(file_path)[1].lower()

if file_extension == ".xes":
    # print(f"📂 Importing XES file: {file_path}")
    with silence_tqdm_and_display(), redirect_stdout(buf_align), redirect_stderr(buf_align):
        # disable pm4py's internal progress bar
        event_log = xes_importer.apply(file_path, parameters={"show_progress_bar": False})
    
    print(f"✅ XES file loaded successfully")

else:
    raise ValueError(f"❌ Unsupported file type: {file_extension}\nPlease provide a .csv or .xes file.")



✅ XES file loaded successfully


### Normalize Activities Names

- pre-processing activity names in both model and log data to assure there's no mismatches

In [ ]:
# normaliztion function for labels
def normalize_label(label):
    """
    Normalize labels by:
    - Converting to lowercase
    - Stripping whitespace
    - Replacing spaces and special chars with "-" (but preserving Portuguese chars like ç, ã, etc.)
    """
    if not isinstance(label, str):
        return ""
    label = label.strip().lower()
    label = label.replace('_', ' ').replace('-', ' ')
    # Replace special punctuation but preserve Portuguese characters
    label = re.sub(r'[^\w\u00C0-\u017F&]+', ' ', label)
    label = re.sub(r'\s+', ' ', label)
    return label.strip()

log_activities_orig = set(event["concept:name"] for trace in event_log for event in trace if "concept:name" in event)
net_activities_orig = set(t.label for t in net.transitions if t.label and t.label not in (None, ">>") and not t.label.startswith("tau"))

all_original_labels = log_activities_orig.union(net_activities_orig)

# create dictionaries for mapping original labels to normalized labels
original_to_normalized = {}
normalized_to_original = {}

for original in all_original_labels:
    normalized = normalize_label(original)
    original_to_normalized[original] = normalized
    normalized_to_original[normalized] = original

# 3. Now perform the normalization using the mapping (more efficient than re-running regex)
print("🔧 Normalizing event log...")
for trace in event_log:
    for event in trace:
        if "concept:name" in event:
            event["concept:name"] = original_to_normalized[event["concept:name"]]

print("🔧 Normalizing Petri net transitions...")
for t in net.transitions:
    if t.label and t.label in original_to_normalized:
        t.label = original_to_normalized[t.label]

print("✅ Event log and Petri net labels normalized.")

# === 4️⃣ Compare activity names and detect mismatches ===
log_activities = set(event["concept:name"] for trace in event_log for event in trace)
net_activities = set(t.label for t in net.transitions if t.label)

model_activities = set(
    t.label for t in net.transitions 
    if t.label and t.label not in (None, ">>") and not t.label.startswith("tau")
)

missing_in_net = log_activities - net_activities
missing_in_log = net_activities - log_activities
mismatch = False


if missing_in_net or missing_in_log:
    mismatch = True
else:
    print("\n✅ All labels match between log and Petri net!")

🔧 Normalizing event log...
🔧 Normalizing Petri net transitions...
✅ Event log and Petri net labels normalized.


In [ ]:
if mismatch:
    print("⚠️ Mismatched labels detected — computing AI-based fuzzy matches ...")

    import re
    from difflib import SequenceMatcher
    from sentence_transformers import SentenceTransformer, util
    from rapidfuzz import fuzz
    import nltk
    from nltk.stem import RSLPStemmer
    import spacy

    # === Load embedding model ===
    model = SentenceTransformer(
        EMBEDDING_MODEL,
        cache_folder="./hf_cache",
        use_auth_token=None
    )

    # === Load Portuguese stemmer and spaCy ===
    nltk.download("rslp", quiet=True)
    stemmer = RSLPStemmer()
    nlp = spacy.load(LINGUISTIC_PIPELINE)

    SEMANTIC_VERB_PENALTY_CONFIG = {
        "high_similarity_floor": 0.75,
        "medium_similarity_floor": 0.55,
        "low_similarity_floor": 0.35,
        "high_similarity_penalty": 0.0,
        "medium_similarity_penalty": 0.05,
        "low_similarity_penalty": 0.15,
        "very_low_similarity_penalty": 0.30,
        "fallback_penalty": 0.0,
    }

    # === Normalization ===
    def normalize(lbl: str) -> str:
        lbl = (lbl or "").lower().strip()
        lbl = lbl.replace("_", " ").replace("-", " ")
        lbl = re.sub(r"[^a-z0-9\u00C0-\u017F ]", "", lbl)
        lbl = re.sub(r"\s+", " ", lbl)
        return lbl

    # === Verb and object extraction ===
    def extract_main_verb(label):
        normalized = normalize(label)
        return normalized.split()[0] if normalized.split() else ""

    def extract_main_object(label):
        tokens = normalize(label).split()[1:]  # skip first verb
        return " ".join(tokens)

    def extract_verbs(label):
        """Extract all verb lemmas using spaCy."""
        doc = nlp(normalize(label))
        return [token.lemma_ for token in doc if token.pos_ == "VERB"]

    def calculate_semantic_verb_penalty(log_label, net_label, action_score):
        try:
            verbs_log = extract_verbs(log_label)
            verbs_net = extract_verbs(net_label)
        except Exception as exc:
            return SEMANTIC_VERB_PENALTY_CONFIG["fallback_penalty"], f"linguistic-fallback:{type(exc).__name__}"

        if not verbs_log or not verbs_net:
            return 0.0, "no-verbs"
        if verbs_log == verbs_net:
            return 0.0, "same-verb-lemmas"
        if action_score >= SEMANTIC_VERB_PENALTY_CONFIG["high_similarity_floor"]:
            return SEMANTIC_VERB_PENALTY_CONFIG["high_similarity_penalty"], "high-action-similarity"
        if action_score >= SEMANTIC_VERB_PENALTY_CONFIG["medium_similarity_floor"]:
            return SEMANTIC_VERB_PENALTY_CONFIG["medium_similarity_penalty"], "medium-action-similarity"
        if action_score >= SEMANTIC_VERB_PENALTY_CONFIG["low_similarity_floor"]:
            return SEMANTIC_VERB_PENALTY_CONFIG["low_similarity_penalty"], "low-action-similarity"
        return SEMANTIC_VERB_PENALTY_CONFIG["very_low_similarity_penalty"], "very-low-action-similarity"

    def get_withholding_reason(confidence_ok, margin_ok, margin_applicable):
        if confidence_ok and margin_ok:
            return None
        reasons = []
        if not confidence_ok:
            reasons.append("confidence-below-theta")
        if margin_applicable and not margin_ok:
            reasons.append("margin-below-epsilon")
        return "+".join(reasons)

    def ranking_key(candidate):
        return (
            -candidate["ranking_score"],
            -candidate["embedding_score"],
            -candidate["verb_score"],
            -candidate["object_score"],
            -candidate["token_score"],
            -candidate["lexical_score"],
            candidate["net_label"],
        )

    # === Prepare label lists ===
    log_labels_list = [lbl for lbl in missing_in_net if lbl and lbl.strip()]
    net_labels_list = sorted([lbl for lbl in net_activities if lbl and lbl.strip()])

    if not log_labels_list:
        print("✅ No mismatched log labels to process.")
    else:
        # === Encode full labels for semantic similarity ===
        log_embeddings = model.encode(log_labels_list, convert_to_tensor=True, show_progress_bar=False)
        net_embeddings = model.encode(net_labels_list, convert_to_tensor=True, show_progress_bar=False)
        full_label_sim = util.cos_sim(log_embeddings, net_embeddings)

        # === Extract verbs and objects ===
        log_verbs = [extract_main_verb(lbl) for lbl in log_labels_list]
        net_verbs = [extract_main_verb(lbl) for lbl in net_labels_list]
        verb_emb_log = model.encode(log_verbs, convert_to_tensor=True, show_progress_bar=False)
        verb_emb_net = model.encode(net_verbs, convert_to_tensor=True, show_progress_bar=False)
        verb_sim = util.cos_sim(verb_emb_log, verb_emb_net)

        log_objects = [extract_main_object(lbl) for lbl in log_labels_list]
        net_objects = [extract_main_object(lbl) for lbl in net_labels_list]
        obj_emb_log = model.encode(log_objects, convert_to_tensor=True, show_progress_bar=False)
        obj_emb_net = model.encode(net_objects, convert_to_tensor=True, show_progress_bar=False)
        obj_sim = util.cos_sim(obj_emb_log, obj_emb_net)

        # === Mapping logic ===
        mapping_suggestions = {}
        non_suggested_mappings = {}
        activity_mapping_rankings = {}
        candidate_rows = []
        threshold = CANDIDATE_THRESHOLD
        minimum_candidate_margin = MINIMUM_CANDIDATE_MARGIN
        total = len(log_labels_list)

        print(f"\n🔄 Progress: 0/{total} (0%)", flush=True)

        # === Compare each log label to all net labels ===
        for i, log_label in enumerate(log_labels_list):
            norm_log = normalize(log_label)
            candidate_scores = []

            for j, net_label in enumerate(net_labels_list):
                norm_net = normalize(net_label)

                # --- 1. Action/verb similarity (semantic) ---
                verb_score = verb_sim[i][j].item()

                # --- 2. Object similarity (semantic) ---
                obj_score = obj_sim[i][j].item()

                # --- 3. Full-label semantic and fuzzy similarities ---
                embedding_score = full_label_sim[i][j].item()
                lev_ratio = SequenceMatcher(None, norm_log, norm_net).ratio()
                token_ratio = fuzz.token_set_ratio(norm_log, norm_net) / 100.0

                # --- 4. Base ranking score h0 ---
                ranking_score = (
                    ACTION_WEIGHT * verb_score +
                    OBJECT_WEIGHT * obj_score +
                    FULL_LABEL_WEIGHT * embedding_score +
                    LEXICAL_WEIGHT * lev_ratio +
                    TOKEN_WEIGHT * token_ratio
                )

                # --- 5. Semantic verb penalty pv and confidence score hc ---
                verb_penalty, verb_penalty_reason = calculate_semantic_verb_penalty(log_label, net_label, verb_score)
                confidence_score = ranking_score - verb_penalty

                candidate_scores.append({
                    "source_activity": log_label,
                    "net_label": net_label,
                    "ranking_score": ranking_score,
                    "confidence_score": confidence_score,
                    "embedding_score": embedding_score,
                    "verb_score": verb_score,
                    "object_score": obj_score,
                    "lexical_score": lev_ratio,
                    "token_score": token_ratio,
                    "verb_penalty": verb_penalty,
                    "verb_penalty_reason": verb_penalty_reason,
                })

            ranked_candidates = sorted(candidate_scores, key=ranking_key)
            top_margin = None
            margin_applicable = len(ranked_candidates) > 1
            if margin_applicable:
                top_margin = ranked_candidates[0]["ranking_score"] - ranked_candidates[1]["ranking_score"]

            for rank_position, candidate in enumerate(ranked_candidates, start=1):
                confidence_ok = candidate["confidence_score"] >= threshold
                margin_ok = True if rank_position != 1 or not margin_applicable else top_margin >= minimum_candidate_margin
                high_confidence = confidence_ok and margin_ok if rank_position == 1 else confidence_ok
                ambiguous = rank_position == 1 and confidence_ok and margin_applicable and not margin_ok
                candidate.update({
                    "rank": rank_position,
                    "candidate_margin": top_margin if rank_position == 1 else None,
                    "margin_applicable": margin_applicable if rank_position == 1 else None,
                    "threshold_condition": confidence_ok,
                    "margin_condition": margin_ok if rank_position == 1 else None,
                    "high_confidence": high_confidence,
                    "ambiguous": ambiguous,
                    "ambiguity_reason": "margin-below-epsilon" if ambiguous else None,
                    "withholding_reason": get_withholding_reason(confidence_ok, margin_ok, margin_applicable) if rank_position == 1 else None,
                })
                candidate_rows.append(candidate)

            activity_mapping_rankings[log_label] = {
                "log_label": log_label,
                "candidates": ranked_candidates,
            }

            best_candidate = ranked_candidates[0] if ranked_candidates else None

            # === Record results ===
            if best_candidate:
                mapping_data = (
                    best_candidate["net_label"],
                    best_candidate["confidence_score"],
                    best_candidate["verb_score"],
                    best_candidate["object_score"],
                    best_candidate["lexical_score"],
                    best_candidate["token_score"],
                    best_candidate["ranking_score"],
                    best_candidate["embedding_score"],
                    best_candidate["verb_penalty"],
                    best_candidate["candidate_margin"],
                    best_candidate["withholding_reason"],
                    best_candidate["ambiguous"],
                    best_candidate["ambiguity_reason"],
                )
                if best_candidate["threshold_condition"]:
                    mapping_suggestions[log_label] = mapping_data
                else:
                    non_suggested_mappings[log_label] = mapping_data
            else:
                non_suggested_mappings[log_label] = ("-", 0, 0, 0, 0, 0, 0, 0, 0, None, "no-candidate", False, None)

            # Print progress
            if (i + 1) % 1 == 0 or (i + 1) == total or (i + 1) % 5 == 0:
                percent = int(100 * (i + 1) / total)
                print(f"🔄 Progress: {i + 1}/{total} ({percent}%)", flush=True)

        # === Print results ===
        if mapping_suggestions:
            print("\n✅ Suggested Mappings (log label → Petri net transition):")
            for src, (tgt, hc, verb, obj, lev, token, h0, full_label, penalty, margin, reason, ambiguous, ambiguity_reason) in sorted(
                mapping_suggestions.items(), key=lambda x: -x[1][1]
            ):
                ambiguity_text = f", ambiguous=True, ambiguity_reason={ambiguity_reason}" if ambiguous else ""
                print(
                    f"  ✅ {src:60s} → {tgt:45s}  "
                    f"(h0={h0:.2f}, hc={hc:.2f}, Full={full_label:.2f}, Verb={verb:.2f}, "
                    f"Obj={obj:.2f}, Lev={lev:.2f}, Token={token:.2f}, pv={penalty:.2f}, "
                    f"Δ={margin if margin is not None else 0:.2f}{ambiguity_text})"
                )
            print(f"\n{len(mapping_suggestions)} mappings satisfy hc >= {threshold}. Near-tie mappings are marked when Delta < {minimum_candidate_margin}.")

        if non_suggested_mappings:
            print("\n❌ Non Suggested Mappings (lower confidence):")
            for src, (tgt, hc, verb, obj, lev, token, h0, full_label, penalty, margin, reason, ambiguous, ambiguity_reason) in sorted(
                non_suggested_mappings.items(), key=lambda x: -x[1][1]
            ):
                print(
                    f"  ❌ {src:60s} → {tgt:45s}  "
                    f"(h0={h0:.2f}, hc={hc:.2f}, Full={full_label:.2f}, Verb={verb:.2f}, "
                    f"Obj={obj:.2f}, Lev={lev:.2f}, Token={token:.2f}, pv={penalty:.2f}, "
                    f"Δ={margin if margin is not None else 0:.2f}, reason={reason})"
                )
            print(f"\n? {len(non_suggested_mappings)} mappings did not satisfy hc ? {threshold}.")

else:
    print("✅ All labels match perfectly — skipping AI-based fuzzy mapping.")

print("\n✅ AI-based fuzzy mapping complete.")


# 📊 Prepare activity lists for manual mapping (available right after AI suggestions)
bpmn_activities_list = sorted(list(net_activities)) if 'net_activities' in locals() else []
log_activities_list = sorted(list(log_activities)) if 'log_activities' in locals() else []

variables_manual_mappings = [
    'bpmn_activities_list',
    'log_activities_list',
]

for _var in variables_manual_mappings:
    if _var in locals():
        try:
            # Trigger potential lazy evaluation (safe for sized containers)
            _ = len(locals()[_var])
        except Exception:
            # If len() is not supported or evaluation fails, ignore silently
            pass


⚠️ Mismatched labels detected — computing AI-based fuzzy matches ...


### Apply New Mappings

- based on the fuzzy matching apply the mapped labels

In [ ]:
#Apply user-confirmed mappings (selected from AI suggestions via frontend)
if 'user_confirmed_mappings' in locals() and user_confirmed_mappings:
    print(f"🔄 Applying {len(user_confirmed_mappings)} user-confirmed mappings to event log...")
    simple_mappings = user_confirmed_mappings
    applied_count = 0
    
    # Apply mappings while preserving the event log structure
    for trace in event_log:
        for event in trace:
            if "concept:name" in event:
                original_label = event["concept:name"]
                # Apply mapping if it exists
                if original_label in simple_mappings:
                    event["concept:name"] = simple_mappings[original_label]
                    applied_count += 1

    print(f"✅ Applied {applied_count} label replacements to event log.")
    print("\n💾 Event log successfully aligned with Petri net labels using user-confirmed mappings.")
else:
    print("✅ No mappings selected — proceeding with original event log labels.")


# if 'mapping_suggestions' not in locals() or not mapping_suggestions:
#     print("✅ No mappings to apply — event log already aligned with Petri net.")
# else:
#     print(f"🔄 Applying {len(mapping_suggestions)} AI-suggested mappings to event log...")

#     # Extract the target mapping (just the target label, not the tuple)
#     simple_mappings = {}
#     for src, mapping_data in mapping_suggestions.items():
#         if isinstance(mapping_data, tuple):
#             simple_mappings[src] = mapping_data[0]  # Extract target from tuple
#         else:
#             simple_mappings[src] = mapping_data

#     applied_count = 0
    
#     # Apply mappings while preserving the event log structure
#     for trace in event_log:
#         for event in trace:
#             if "concept:name" in event:
#                 original_label = event["concept:name"]
#                 # Apply mapping if it exists
#                 if original_label in simple_mappings:
#                     event["concept:name"] = simple_mappings[original_label]
#                     applied_count += 1
    
#     for trace in event_log:
#         for event in trace:
#             if "concept:name" in event:
#                 original_label = event["concept:name"]
#                 # Apply mapping if it exists
#                 if original_label in simple_mappings:
#                     event["concept:name"] = simple_mappings[original_label]
#                     applied_count += 1

#     print(f"✅ Applied {applied_count} label replacements to event log.")
#     print(f"📊 Mappings used:")
#     for src, tgt in simple_mappings.items():
#         print(f"   {src} → {tgt}")

#     print("💾 Event log successfully aligned with Petri net labels.")

🔄 Applying 1 AI-suggested mappings to event log...
✅ Applied 4 label replacements to event log.
📊 Mappings used:
   send-for-authorization → send-for-final-expenditure-authorization
💾 Event log successfully aligned with Petri net labels.


### Utilitary Functions

In [ ]:
import copy
from pm4py.objects.petri_net.obj import PetriNet, Marking
from collections import defaultdict

def get_next_model_activities(petri_net: PetriNet, initial_marking: Marking, final_marking: Marking):
    """
    Returns a dictionary of visible transitions and their possible next visible transitions,
    traversing invisible transitions (gateways) automatically.
    Includes:
        - <<start>>: as starting point of the process
        - <<end>>: as possible next activity for transitions that reach final marking
    """
    next_activities = defaultdict(set)
    
    def traverse(place):
        """Return all visible transitions reachable from this place through invisible transitions.
        Also detects if the place is in final marking and adds <<end>>."""
        result = set()
        
        # Check if this place is in the final marking
        if place in final_marking:
            result.add("<<end>>")
        
        for arc in place.out_arcs:
            t = arc.target
            if t.label is not None:
                result.add(t.label)
            else:
                # invisible transition, go to its output places recursively
                for out_arc in t.out_arcs:
                    result |= traverse(out_arc.target)
        return result
    
    # Compute next activities for each visible transition
    for t in petri_net.transitions:
        if t.label is not None:
            next_set = set()
            for place_arc in t.out_arcs:
                next_set |= traverse(place_arc.target)
            next_activities[t.label].update(next_set)
    
    # Add <<start>> activity pointing to all visible transitions enabled at initial marking
    start_next = set()
    for place, tokens in initial_marking.items():
        if tokens > 0:
            start_next |= traverse(place)
    next_activities["<<start>>"] = start_next
    
    # Convert sets to lists
    return {k: list(v) for k, v in next_activities.items()}

def extract_activity(side):
    """
    Extract activity label safely from alignment side.
    Works for:
    - 'A'
    - '>>'
    - ('t_A', 'A')
    - None
    """
    if side is None:
        return None

    # If side is already a string
    if isinstance(side, str):
        if side == ">>":
            return None
        return side

    # If side is a tuple like ('t_A', 'A')
    if isinstance(side, tuple):
        if len(side) > 1:
            return side[1]
        return None

    return None

def compute_full_reachability(next_map):
    """
    Computes all reachable visible activities from each activity,
    following all next_activity edges recursively.
    """
    full = {a: set() for a in next_map}

    for a in next_map:
        visited = set()
        stack = list(next_map[a])  # start from immediate successors

        while stack:
            x = stack.pop()
            if x not in visited:
                visited.add(x)
                full[a].add(x)
                if x in next_map:
                    stack.extend(next_map[x])

    return full

def is_model_reachable(prev_model_activity, current_log_activity, model_activities, model_reachable):
    """
    Checks whether the log activity can follow the previous model activity
    according to the Petri-net reachability (silent transitions allowed).
    """
    # If we have no reachability info for the previous activity → no
    if prev_model_activity not in model_reachable:
        return False

    # If log activity does not exist in the model at all → deviation
    if current_log_activity not in model_activities:
        return False

    # If reachable through silent/gateway transitions → valid
    return current_log_activity in model_reachable[prev_model_activity]

def preprocess_event_log(log):
    fixed_log = EventLog()

    for trace in log:
        new_trace = Trace(attributes=copy.deepcopy(trace.attributes))
        events_by_activity = defaultdict(list)

        for idx, event in enumerate(trace):
            activity = event.get("concept:name")
            transition = str(event.get("lifecycle:transition", "")).lower().strip()
            events_by_activity[activity].append((idx, transition, event))

        events_to_add = []

        for _, items in events_by_activity.items():
            pending_starts = []
            unmatched_completes = []

            # Match start/complete in trace order for each activity
            for idx, transition, event in items:
                if transition == "start":
                    pending_starts.append((idx, event))
                elif transition == "complete":
                    if pending_starts:
                        start_idx, start_event = pending_starts.pop(0)
                        events_to_add.append(((start_idx, 0), copy.deepcopy(start_event)))
                        events_to_add.append(((idx, 1), copy.deepcopy(event)))
                    else:
                        unmatched_completes.append((idx, event))
                else:
                    events_to_add.append(((idx, 0), copy.deepcopy(event)))

            # Handle unmatched starts
            for start_idx, start_event in pending_starts:
                ts = start_event.get("time:timestamp")
                if ts is None:
                    continue
                start_ev = copy.deepcopy(start_event)
                start_ev["lifecycle:transition"] = "start"
                start_ev["time:timestamp"] = ts
                complete_ev = copy.deepcopy(start_event)
                complete_ev["lifecycle:transition"] = "complete"
                complete_ev["time:timestamp"] = ts
                events_to_add.append(((start_idx, 0), start_ev))
                events_to_add.append(((start_idx, 1), complete_ev))

            # Handle unmatched completes
            for comp_idx, comp_event in unmatched_completes:
                ts = comp_event.get("time:timestamp")
                if ts is None:
                    continue
                start_ev = copy.deepcopy(comp_event)
                start_ev["lifecycle:transition"] = "start"
                start_ev["time:timestamp"] = ts
                complete_ev = copy.deepcopy(comp_event)
                complete_ev["lifecycle:transition"] = "complete"
                complete_ev["time:timestamp"] = ts
                events_to_add.append(((comp_idx, -1), start_ev))
                events_to_add.append(((comp_idx, 0), complete_ev))

        for _, event in sorted(events_to_add, key=lambda x: x[0]):
            new_trace.append(event)

        if len(new_trace) > 0:
            fixed_log.append(new_trace)

    return fixed_log

def count_mismatched_transitions(log):
    missing_start = 0
    missing_complete = 0

    for trace in log:
        activity_groups = {}

        for event in trace:
            activity = event["concept:name"]
            activity_groups.setdefault(activity, []).append(event)

        for _, events in activity_groups.items():
            transitions = [e["lifecycle:transition"].lower() for e in events]

            has_start = "start" in transitions
            has_complete = "complete" in transitions

            if has_start and not has_complete:
                missing_complete += 1
            elif has_complete and not has_start:
                missing_start += 1

    total_mismatches = missing_start + missing_complete

    return missing_start, missing_complete, total_mismatches

def count_events_without_timestamp(log):
    total_events = 0
    missing_timestamp = 0

    for trace in log:
        for event in trace:
            total_events += 1
            if "time:timestamp" not in event or event["time:timestamp"] is None:
                missing_timestamp += 1

    return total_events, missing_timestamp

def normalize_frontend_key(label):
    label = (label or "").strip().lower()
    label = label.replace("_", "-")
    label = re.sub(r"[^\w\u00C0-\u017F&-]+", "-", label)
    label = re.sub(r"-+", "-", label)
    return label.strip("-")

### Log Pre-Processing

- since there can be multiple activities with incomplete lifecycle transitions there's a need to pre-process the event log:
    - find which activities have only one of the transitions:
		- has timestamp -> create the missing transition with the same timestamp
		- no timestamp -> remove the transition (activity)

- creation of a log with only one transition for conformance metrics

In [ ]:
from pm4py.objects.log.obj import EventLog, Trace
from pm4py.objects.log.util import interval_lifecycle

event_log_fixed = preprocess_event_log(event_log)

start_activities = pm4py.get_start_activities(event_log_fixed)
end_activities = pm4py.get_end_activities(event_log_fixed)
interval_log = interval_lifecycle.to_interval(event_log_fixed)
df_pm = pm4py.convert_to_dataframe(interval_log)

event_log_complete = EventLog()

for trace in event_log_fixed:
    new_trace = Trace(attributes=trace.attributes)
    for event in trace:
        if event.get("lifecycle:transition") == "complete":
            new_trace.append(event)
    if len(new_trace) > 0:
        event_log_complete.append(new_trace)

### Generic PM4PY Process Metrics

- alignments
- conformance profile

In [ ]:
from pm4py.algo.conformance.alignments.petri_net import algorithm as align_algo
from pm4py.algo.evaluation.replay_fitness import algorithm as fitness_eval
from pm4py.algo.evaluation.precision import algorithm as precision_eval
from pm4py.algo.evaluation.generalization import algorithm as generalization_eval
from pm4py.algo.evaluation.simplicity import algorithm as simplicity_eval
from pm4py.algo.conformance.tokenreplay import algorithm as token_replay
from collections import Counter

buf_align = io.StringIO()

# Helper: context manager to silence tqdm and IPython display outputs
from contextlib import contextmanager
@contextmanager
def silence_tqdm_and_display():
    try:
        import tqdm
        from IPython import display as ipdisplay
    except Exception:
        # If imports fail, act as a no-op context manager
        yield
        return

    _tqdm_orig = getattr(tqdm, 'tqdm', None)
    _tqdm_nb_orig = getattr(getattr(tqdm, 'notebook', None), 'tqdm', None) if hasattr(tqdm, 'notebook') else None
    _ipydisplay_orig = getattr(ipdisplay, 'display', None)

    def _silent_tqdm(*a, **k):
        # Force disable progress bars
        k['disable'] = True
        if _tqdm_orig:
            return _tqdm_orig(*a, **k)
        # Fallback: return the iterable if provided
        return a[0] if a else iter([])

    try:
        if _tqdm_orig:
            tqdm.tqdm = _silent_tqdm
        if hasattr(tqdm, 'notebook') and _tqdm_nb_orig:
            tqdm.notebook.tqdm = _silent_tqdm
        ipdisplay.display = lambda *a, **k: None
        yield
    finally:
        try:
            if _tqdm_orig:
                tqdm.tqdm = _tqdm_orig
            if _tqdm_nb_orig and hasattr(tqdm, 'notebook'):
                tqdm.notebook.tqdm = _tqdm_nb_orig
        except Exception:
            pass
        if _ipydisplay_orig:
            ipdisplay.display = _ipydisplay_orig

print("🚀 Starting comprehensive conformance checking between event log and Petri net...")

# ---------- 1️⃣ Run Alignments (A* Algorithm) ----------
params = {
    "max_align_time": 10,  # limit per trace (seconds)
    "max_cost": 1000,    # optional cost cap for large logs
    "show_progress_bar": False  # disable progress bar
}

with silence_tqdm_and_display(), redirect_stdout(buf_align), redirect_stderr(buf_align):
    aligned_traces = align_algo.apply_log(event_log_complete, net, im, fm, parameters=params)

# ---------- 2️⃣ Alignment Fitness Summary ----------
trace_fitness = [res.get("fitness", 0.0) for res in aligned_traces]
mean_fitness = float(numpy.mean(trace_fitness))
fit_traces_pct = 100.0 * numpy.mean([f >= 1.0 for f in trace_fitness])

# print("\n📈 Alignment Summary:")
# print(f"   • Mean fitness: {mean_fitness:.4f}")
# print(f"   • % of perfectly fitting traces: {fit_traces_pct:.2f}%")

# ---------- 3️⃣ Comprehensive Conformance Metrics ----------
#print("\n🧮 Computing comprehensive conformance metrics...")

# Fitness (already computed via alignments)
fitness_score = mean_fitness

# Precision - how much behavior in the model is observed in the log
#print("   • Computing precision...")
with silence_tqdm_and_display(), redirect_stdout(buf_align), redirect_stderr(buf_align):
    precision_score = precision_eval.apply(event_log_complete, net, im, fm, variant=precision_eval.Variants.ALIGN_ETCONFORMANCE, parameters={"show_progress_bar": False})

# Generalization - how well the model generalizes beyond the observed behavior
#print("   • Computing generalization...")
with silence_tqdm_and_display(), redirect_stdout(buf_align), redirect_stderr(buf_align):
    generalization_score = generalization_eval.apply(event_log_complete, net, im, fm)

# Simplicity - how simple/complex the model is (lower complexity = higher simplicity)
#print("   • Computing simplicity...")
with silence_tqdm_and_display(), redirect_stdout(buf_align), redirect_stderr(buf_align):
    simplicity_score = simplicity_eval.apply(net)

# Create comprehensive conformance profile
conformance_profile = {
    "fitness": fitness_score,
    "precision": precision_score,
    "generalization": generalization_score,
    "simplicity": simplicity_score
}

# print("\n🎯 Comprehensive Conformance Profile:")
# print("=" * 50)
# for metric, score in conformance_profile.items():
#     print(f"   {metric.capitalize():15s}: {score:.4f}")

# Overall quality score (weighted average)
# overall_quality = (0.4 * fitness_score + 0.3 * precision_score + 0.2 * generalization_score + 0.1 * simplicity_score)
# print(f"   {'Overall Quality':15s}: {overall_quality:.4f}")
# print("=" * 50)

# Quality interpretation
# print("\n📊 Quality Interpretation:")
# def interpret_score(score, metric_name):
#     if score >= 0.8:
#         return f"🟢 Excellent {metric_name}"
#     elif score >= 0.6:
#         return f"🟡 Good {metric_name}"
#     elif score >= 0.4:
#         return f"🟠 Fair {metric_name}"
#     else:
#         return f"🔴 Poor {metric_name}"

# for metric, score in conformance_profile.items():
#     print(f"   {interpret_score(score, metric)}")

# ---------- 4️⃣ Log-level Fitness (Official PM4Py Metric) ----------
with silence_tqdm_and_display(), redirect_stdout(buf_align), redirect_stderr(buf_align):
    fit = fitness_eval.apply(event_log_complete, net, im, fm, variant=fitness_eval.Variants.ALIGNMENT_BASED)

# print("\n🔬 Detailed Fitness Metrics:")
# for k, v in fit.items():
#     print(f"   {k}: {v:.4f}" if isinstance(v, (int, float)) else f"   {k}: {v}")

# ---------- 5️⃣ Analyze Deviations ----------
def deviating_steps(aln_dict):
    dev = []
    for a, b in aln_dict["alignment"]:
        if a == ">>" and b != ">>":      # model move (missing in log)
            dev.append(("model", b))
        elif a != ">>" and b == ">>":    # log move (unexpected in model)
            dev.append(("log", a))
    return dev

counts = Counter()
for r in aligned_traces:
    counts.update(deviating_steps(r))

# print("\n🚨 Top Deviations (most frequent mismatches):")
# for (kind, act), c in counts.most_common(10):
#     print(f"   {kind.upper():5s} {act}: {c}")

# ---------- 6️⃣ Inspect One Example Alignment ----------
# example_idx = 0
# aln = aligned_traces[example_idx]["alignment"]
# print(f"\n🔍 Example Alignment (trace {example_idx}):")
# for a, b in aln: 
#     a_label = a if a != '>>' else '(No Label)'
#     b_label = b if b != '>>' else '(No Label)'
#     print(f"   {a_label}  ⟂  {b_label}")

# ---------- 7️⃣ Token-Based Replay (Alternative Sanity Check) ----------
with silence_tqdm_and_display(), redirect_stdout(buf_align), redirect_stderr(buf_align):
    tbr = token_replay.apply(event_log_complete, net, im, fm, parameters={"show_progress_bar": False})
tbr_fitness = float(numpy.mean([x["trace_fitness"] for x in tbr]))
#print(f"\nMean Token-Based Replay fitness: {tbr_fitness:.4f}")

print("\n✅ Comprehensive conformance checking completed successfully.")


🚀 Starting comprehensive conformance checking between event log and Petri net...


replaying log with TBR, completed traces ::   0%|          | 0/75 [00:00<?, ?it/s]

aligning log, completed variants ::   0%|          | 0/75 [00:00<?, ?it/s]


✅ Comprehensive conformance checking completed successfully.


### Roles & Lanes Helpers

- roles and frequencies per activity
- lanes and frequencies per role

In [ ]:
from pm4py.algo.filtering.log.attributes import attributes_filter
from collections import defaultdict, Counter
import json

print("📊 Extracting activity roles from log and matching with BPMN lanes...")

model_next_activities = get_next_model_activities(net, im, fm)

activity_role_frequencies = df_pm.groupby(['concept:name', 'papel']).size().to_dict()

activities_roles_all = {}
for (activity, role), freq in activity_role_frequencies.items():
    if activity not in activities_roles_all:
        activities_roles_all[activity] = []
    activities_roles_all[activity].append({"role": role, "frequency": freq})

from collections import defaultdict

# model-based role frequencies
activity_role_frequencies_model = defaultdict(lambda: defaultdict(int))

# reconstruct the case sequences and check against the model
for case_id, case_df in df_pm.groupby("case:concept:name"):

    case_df = case_df.sort_values("time:timestamp")

    prev_activity = "<<start>>"

    for _, row in case_df.iterrows():
        activity = row["concept:name"]
        role = row["papel"]

        # check if the flow respects the model
        if activity in model_next_activities.get(prev_activity, []):
            activity_role_frequencies_model[activity][role] += 1

        prev_activity = activity

    # handle end transition (optional, usually not needed for roles)
    if "<<end>>" in model_next_activities.get(prev_activity, []):
        pass

activities_roles_model = {}

for activity, roles in activity_role_frequencies_model.items():
    activities_roles_model[activity] = [
        {"role": role, "frequency": freq}
        for role, freq in roles.items()
    ]

activities_roles = {
    "model": activities_roles_model,
    "all": activities_roles_all
}

print("📌 Extracting lanes and activities from BPMN model...")

# reuse the BPMN XML tree and namespaces loaded earlier in the pipeline.
if "bpmn_tree" not in globals() or "namespaces" not in globals():
    raise RuntimeError("BPMN XML tree not loaded. Run the BPMN import and cycle-time extraction cells first.")

# extract lanes and their flowNodeRef (activities)
lanes_with_activities = {}

for lane in bpmn_tree.xpath('//bpmn:lane', namespaces=namespaces):
    lane_id = lane.get('id')
    lane_name = lane.get('name', lane_id)
    
    # get all flowNodeRef elements (activities in this lane)
    flow_node_refs = lane.xpath('./bpmn:flowNodeRef', namespaces=namespaces)
    activity_ids = [ref.text for ref in flow_node_refs if ref.text]
    
    lanes_with_activities[lane_id] = {
        'lane_name': lane_name,
        'activity_ids': activity_ids,
        'activities': []
    }

# map activity IDs to activity names
activity_id_to_name = {}
for task in bpmn_tree.xpath('//bpmn:task | //bpmn:userTask | //bpmn:serviceTask | //bpmn:subProcess', namespaces=namespaces):
    task_id = task.get('id')
    task_name = task.get('name')
    if task_id and task_name:
        normalized_name = normalize_label(task_name)
        activity_id_to_name[task_id] = normalized_name

# populate activity names for each lane
for lane_id, lane_data in lanes_with_activities.items():
    for activity_id in lane_data['activity_ids']:
        if activity_id in activity_id_to_name:
            lane_data['activities'].append(activity_id_to_name[activity_id])

# create activity to lane mapping
activity_to_lane = {}
for lane_id, lane_data in lanes_with_activities.items():
    for activity in lane_data['activities']:
        activity_to_lane[activity] = {
            'lane_id': lane_id,
            'lane_name': lane_data['lane_name']
        }

roles_lanes = {}

# build role-to-lanes mapping based on activities and their roles
for activity, roles_list in activities_roles_all.items():
    # get the lane info for this activity
    lane_info = activity_to_lane.get(activity)
    
    if lane_info:
        lane_id = lane_info['lane_id']
        lane_name = lane_info['lane_name']
        
        # for each role that performed this activity
        for role_data in roles_list:
            role = role_data['role']
            frequency = role_data['frequency']
            
            if role not in roles_lanes:
                roles_lanes[role] = {}
            
            # add or update lane frequency for this role
            if lane_id not in roles_lanes[role]:
                roles_lanes[role][lane_id] = {
                    'lane_name': lane_name,
                    'frequency': 0
                }
            roles_lanes[role][lane_id]['frequency'] += frequency

# convert to final structure - lanes directly under role name
roles_lanes = {
    role: [
        {
            'lane_id': lane_id,
            'lane_name': lane_data['lane_name'],
            'frequency': lane_data['frequency']
        }
        for lane_id, lane_data in sorted(
            lanes_dict.items(),
            key=lambda x: (-x[1]['frequency'], x[0])
        )
    ]
    for role, lanes_dict in sorted(roles_lanes.items())
}

print(f"✅ Created role-to-lanes mapping for {len(roles_lanes)} unique roles")

📊 Extracting activity roles from log and matching with BPMN lanes...
📌 Model previous activities (for reachability checks):
  analyze-compliance-with-state-budget-law ← ['analyze-purchase-request', 'authorize-multi-year-acquisition']
  register-in-dm-and-notify-proponent ← ['send-purchase-order']
  de-obligate-expenditure ← ['evaluate-and-grant-final-expenditure-authorization']
  submit-additional-budget-justification ← ['analyze-purchase-request']
  send-purchase-order ← ['prepare-send-purchase-order-contract']
  prepare-purchase-process-draft-contract ← ['analyze-purchase-request', 'authorize-multi-year-acquisition', 'budget-adjustment']
  send-for-final-expenditure-authorization ← ['finalize-budget-allocation-and-send-for-authorization']
  assign-to-project ← ['prepare-purchase-process-draft-contract']
  authorize-multi-year-acquisition ← ['analyze-purchase-request']
  issue-opinion ← ['submit-request']
  submit-request ← ['<<start>>']
  validate-data-and-verify-funding-availability

### Performance Metrics

- compute sojourn time of activities and flows using DFG

In [ ]:
START_NODE = "<<start>>"
END_NODE = "<<end>>"

# variables to hold the flows for performance and frequency

flows_start_perf = [
    {
        "from": START_NODE,
        "to": act,
        "avg_sojourn": 0.0,
        "median_sojourn": 0.0,
    }
    for act in start_activities.keys()
]

flows_start_freq = [
    {
        "from": START_NODE,
        "to": act,
        "frequency": int(count)
    }
    for act, count in start_activities.items()
]

flows_end_perf = [
    {
        "from": act,
        "to": END_NODE,
        "avg_sojourn": 0.0,
        "median_sojourn": 0.0,
    }
    for act in end_activities.keys()
]

flows_end_freq = [
    {
        "from": act,
        "to": END_NODE,
        "frequency": int(count)
    }
    for act, count in end_activities.items()
]


In [ ]:
from pm4py.visualization.dfg import visualizer as dfg_visualizer
from pm4py.algo.discovery.dfg import algorithm as dfg_discovery
from pm4py.algo.discovery.dfg.variants import performance as perf_variant
from pm4py.statistics.start_activities.log import get as sa_get
from pm4py.statistics.end_activities.log import get as ea_get

flow_model_activities = set(model_activities)
flow_model_activities.add("<<start>>")
flow_model_activities.add("<<end>>")

# discover performance DFG

params = {
    perf_variant.Parameters.ACTIVITY_KEY: "concept:name",
    perf_variant.Parameters.TIMESTAMP_KEY: "time:timestamp",
    perf_variant.Parameters.START_TIMESTAMP_KEY: "start_timestamp",
    perf_variant.Parameters.AGGREGATION_MEASURE: "mean",
}

dfg_perf = dfg_discovery.apply(
    interval_log,
    variant=dfg_discovery.Variants.PERFORMANCE,
    parameters=params
)

#print("Edges:", len(dfg_perf))
#list(dfg_perf.items())[:5]

# Calculate sojourn times

# execution time (sojourn) per event (seconds)
df_pm["soj_sec"] = (df_pm["time:timestamp"] - df_pm["start_timestamp"]).dt.total_seconds()

# mean sojourn time per activity
soj_time = df_pm.groupby("concept:name")["soj_sec"].mean().to_dict()

# === logic to show the DFG === 

# Show a few
# for a in list(soj_time.keys())[:10]:
#     print(a, "=>", f"{soj_time[a]:.2f} sec")

# viz_params = {
#     "format": "png",
#     "aggregationMeasure": "mean",
#     "start_activities": start_activities,
#     "end_activities": end_activities,
#     # "max_no_of_edges_in_diagram": 200,  # uncomment if needed
# }

# gviz = dfg_visualizer.apply(
#     dfg_perf,
#     log=interval_log,
#     variant=dfg_visualizer.Variants.PERFORMANCE,
#     parameters=viz_params,
#     serv_time=soj_time
# )

# dfg_visualizer.view(gviz)

# compute activity statistics for all log activities (not just model activities)
activity_stats = df_pm.groupby("concept:name")["soj_sec"].agg(['mean', 'median', 'min','max','count']).reset_index()

activities = {
    row["concept:name"]: {
        "avg_sojourn": round(row["mean"], 3),
        "median_sojourn": round(row["median"], 3),
        "min_sojourn": round(row["min"], 3),
        "max_sojourn": round(row["max"], 3),
        "event_count": int(row["count"]),
        "inModel": row["concept:name"] in model_activities,
        "originalActivity": normalized_to_original.get(row["concept:name"], row["concept:name"])
    }
    for _, row in activity_stats.iterrows()
}

activities_deviations = {
    act: info for act, info in activities.items()
    if not info["inModel"]
}

valid_event_indexes = set()

for case_id, case_df in df_pm.groupby("case:concept:name"):

    case_df = case_df.sort_values("time:timestamp")
    prev_activity = "<<start>>"

    for idx, row in case_df.iterrows():
        activity = row["concept:name"]

        # keep event only if transition is valid in the model
        if activity in model_next_activities.get(prev_activity, []):
            valid_event_indexes.add(idx)

        prev_activity = activity

df_pm_model = df_pm.loc[list(valid_event_indexes)]

activity_stats_model = (
    df_pm_model
    .groupby("concept:name")["soj_sec"]
    .agg(['mean', 'median', 'min', 'max', 'count'])
    .reset_index()
)

# initialize all model activities with zeros
activities_model = {
    act: {
        "avg_sojourn": 0,
        "median_sojourn": 0,
        "min_sojourn": 0,
        "max_sojourn": 0,
        "event_count": 0,
        "inModel": True,
        "originalActivity": normalized_to_original.get(act, act)
    }
    for act in model_activities
}

# overwrite with actual statistics where available
for _, row in activity_stats_model.iterrows():
    act = row["concept:name"]

    activities_model[act] = {
        "avg_sojourn": round(row["mean"], 3),
        "median_sojourn": round(row["median"], 3),
        "min_sojourn": round(row["min"], 3),
        "max_sojourn": round(row["max"], 3),
        "event_count": int(row["count"]),
        "inModel": True,
        "originalActivity": normalized_to_original.get(act, act)
    }


dfg_counts = dfg_discovery.apply(
    interval_log,
    variant=dfg_discovery.Variants.FREQUENCY  # counts, not performance
)

# viz_params_freq = {
#     "format": "png",
#     "start_activities": start_activities,
#     "end_activities": end_activities,
#     "image_format": "png"
# }

# gviz = dfg_visualizer.apply(
#     dfg_counts,
#     log=interval_log,
#     variant=dfg_visualizer.Variants.PERFORMANCE,
#     parameters=viz_params_freq,
#     serv_time=soj_time
# )

# dfg_visualizer.view(gviz)

# compute flow statistics (waiting times between activities)

# sort by case and timestamp to compute next activity and waiting time
df_flow = df_pm.sort_values(["case:concept:name", "time:timestamp"]).copy()

df_flow["next_activity"] = (
    df_flow
    .groupby("case:concept:name")["concept:name"]
    .shift(-1)
)

df_flow["next_start_timestamp"] = (
    df_flow
    .groupby("case:concept:name")["start_timestamp"]
    .shift(-1)
)

df_flow["flow_waiting_time"] = (
    df_flow["next_start_timestamp"] - df_flow["time:timestamp"]
).dt.total_seconds()

# remove invalid/overlapping negative waits
df_flow = df_flow[df_flow["flow_waiting_time"] >= 0]

flow_stats = (
    df_flow
    .dropna(subset=["next_activity", "flow_waiting_time"])
    .groupby(["concept:name", "next_activity"])["flow_waiting_time"]
    .agg(["mean", "median", "count"])
    .reset_index()
)

flows_performance = [
    {
        "from": row["concept:name"],
        "to": row["next_activity"],
        "avg_sojourn": round(row["mean"], 3),
        "median_sojourn": round(row["median"], 3),
    }
    for _, row in flow_stats.iterrows()
]

flows_performance = flows_start_perf + flows_performance + flows_end_perf

# compute flows that are valid according to the model
flows_model_performance = []

for flow in flows_performance:
    src = flow.get("from")
    tgt = flow.get("to")
    if src in flow_model_activities and tgt in flow_model_activities:
        if src in model_next_activities and tgt in model_next_activities.get(src, []):
            flows_model_performance.append({
                "from": src,
                "to": tgt,
                "avg_sojourn": round(flow.get("avg_sojourn", 0), 3),
                "median_sojourn": round(flow.get("median_sojourn", 0), 3)
            })

# compute the set of all model flows
flows_model_set = {(f["from"], f["to"]) for f in flows_model_performance}
flows_deviations_performance = [
    flow for flow in flows_performance
    if (flow["from"], flow["to"]) not in flows_model_set
]

# compute flow frequencies
trace_durations = [
    (trace[-1]["time:timestamp"] - trace[0]["start_timestamp"]).total_seconds()
    for trace in interval_log
]

general_stats = {
    "avg_trace_time": round(pandas.Series(trace_durations).mean(), 3),
    "min_trace_time": round(min(trace_durations), 3),
    "max_trace_time": round(max(trace_durations), 3),
    "total_traces": len(trace_durations),
    "total_events": len(df_pm)
}

performance_metrics = {
    "activities": {
        "model": activities_model,
        "deviations": activities_deviations,
        "all": activities
    },
    "flows": {
        "model": flows_model_performance,
        "deviations": flows_deviations_performance,
        "all": flows_performance
    },
    "general": general_stats
}

print("\n ✅ Performance metrics computed successfully.")


 ✅ Performance metrics computed successfully.


In [ ]:
# build time conformance metrics based on cycle times and observed waiting times

SECONDS_PER_DAY = 24 * 60 * 60

# convert cycle time from days to seconds
def cycle_time_to_seconds(activity):
    if activity in (START_NODE, END_NODE):
        return 0

    cycle_time_days = cycle_times_activities.get(activity, 0)
    return round(float(cycle_time_days) * SECONDS_PER_DAY, 3)

# compute the target time for a given activity
def target_activity_time(activity):
    if activity in (START_NODE, END_NODE):
        return 0

    return round(
        performance_metrics["activities"]["model"]
        .get(activity, {})
        .get("avg_sojourn", 0),
        3,
    )

# compute the median target time for a given activity
def median_activity_time(activity):
    if activity in (START_NODE, END_NODE):
        return 0

    return round(
        performance_metrics["activities"]["model"]
        .get(activity, {})
        .get("median_sojourn", 0),
        3,
    )

# build time conformance metrics from flow data
def build_time_conformance_from_flows(flows):
    result = []

    for flow in flows:
        source = flow.get("from")
        target = flow.get("to")

        avg_waiting_time = round(flow.get("avg_sojourn", 0), 3)
        median_waiting_time = round(flow.get("median_sojourn", flow.get("avg_sojourn", 0)), 3)

        result.append({
            "from": source,
            "to": target,
            "avg_waiting_time": avg_waiting_time,
            "median_waiting_time": median_waiting_time,
            "avg_target_time": target_activity_time(target),
            "median_target_time": median_activity_time(target),
            "cycle_time": cycle_time_to_seconds(target),
        })

    return result

# build time conformance metrics for model, deviations, and all flows
if cycle_times_activities:
    time_conformance = {
        "model": build_time_conformance_from_flows(performance_metrics["flows"]["model"]),
        "deviations": build_time_conformance_from_flows(performance_metrics["flows"]["deviations"]),
        "all": build_time_conformance_from_flows(performance_metrics["flows"]["all"]),
    }
else:
    time_conformance = {
        "model": [],
        "deviations": [],
        "all": [],
    }

print("\n ✅ Conformance performance metrics computed successfully.")

NameError: name 'performance_metrics' is not defined

### Frequency Related Metrics

- compute frequencies for activities and flows using DFG

In [ ]:
from pm4py.statistics.attributes.log import get as attr_get
from collections import defaultdict

# frequencies of all activities in the log
activities_frequencies = {
    act: {
        "frequency": freq,
        "inModel": act in model_activities,
        "originalActivity": normalized_to_original.get(act, act)
    }
    for act, freq in attr_get.get_attribute_values(interval_log, "concept:name").items()
}

# frequencies of deviation activities
activities_frequencies_deviations = {
    act: data for act, data in activities_frequencies.items() 
    if not data["inModel"]
}

activities_moves = defaultdict(lambda: {
    "model_moves": 0,
    "log_moves": 0,
    "synchronous_moves": 0
})

flows_frequency = [
    {
        "from": src,
        "to": tgt,
        "frequency": dfg_counts.get((src, tgt), 0)
    }
    for (src, tgt) in set(list(dfg_perf.keys()) + list(dfg_counts.keys()))
]

flows_frequency = flows_start_freq + flows_frequency + flows_end_freq

# compute flows that are valid according to the model
flows_model_frequency = []

for flow in flows_frequency:
    src = flow.get("from")
    tgt = flow.get("to")
    if src in flow_model_activities and tgt in flow_model_activities:
        if src in model_next_activities and tgt in model_next_activities.get(src, []):
            flows_model_frequency.append({
                "from": src,
                "to": tgt,
                "frequency": flow.get("frequency", 0)
            })

# compute the set of all model flows for frequency
flows_model_set = {(f["from"], f["to"]) for f in flows_model_frequency}
flows_deviations_frequency = [
    flow for flow in flows_frequency
    if (flow["from"], flow["to"]) not in flows_model_set
]

incoming = defaultdict(int)
outgoing = defaultdict(int)

for flow in flows_model_frequency:
    src = flow["from"]
    tgt = flow["to"]
    freq = flow.get("frequency", 0)

    outgoing[src] += freq
    incoming[tgt] += freq


model_activity_counts = {}

for act in flow_model_activities:

    if act in ["<<start>>", "<<end>>"]:
        continue

    in_freq = incoming.get(act, 0)
    out_freq = outgoing.get(act, 0)

    model_activity_counts[act] = {
        "frequency": in_freq,
        "mismatch_incoming": max(out_freq - in_freq, 0),
        "mismatch_outgoing": max(in_freq - out_freq, 0)
    }


activities_frequencies_model = {
    act: {
        "frequency": data["frequency"],
        "inModel": True,
        "originalActivity": normalized_to_original.get(act, act),
        "mismatch_incoming": data["mismatch_incoming"],
        "mismatch_outgoing": data["mismatch_outgoing"]
    }
    for act, data in model_activity_counts.items()
}

frequency_metrics = {
    "activities": {
        "model": activities_frequencies_model,
        "deviations": activities_frequencies_deviations,
        "all": activities_frequencies
    },
    "flows": {
        "model": flows_model_frequency,
        "deviations": flows_deviations_frequency,
        "all": flows_frequency
    },
}

print("📈 Deviation activity frequencies computed")

📈 Deviation activity frequencies computed


### Activity Related Metrics

- frequency of only model activities & log activities
- log/model/sync moves for each model activity

In [ ]:
print("📊 Computing model moves, log moves, and percentages per activity...")

# compute model moves, log moves, and synchronous moves per activity
for trace in aligned_traces:
    for move in trace["alignment"]:

        log_side, model_side = move

        model_act = extract_activity(model_side)
        log_act = extract_activity(log_side)

        # Synchronous
        if model_act and log_act:
            activities_moves[model_act]["synchronous_moves"] += 1

        # Log move
        elif log_act and not model_act:
            activities_moves[log_act]["log_moves"] += 1

        # Model move
        elif model_act and not log_act:
            activities_moves[model_act]["model_moves"] += 1

# compute percentages
for activity, data in activities_moves.items():
    total = sum([
        data["model_moves"],
        data["log_moves"],
        data["synchronous_moves"]
    ])

    if total > 0:
        data["model_moves_percentage"] = 100 * data["model_moves"] / total
        data["log_moves_percentage"] = 100 * data["log_moves"] / total
        data["synchronous_moves_percentage"] = 100 * data["synchronous_moves"] / total
    else:
        data["model_moves_percentage"] = 0
        data["log_moves_percentage"] = 0
        data["synchronous_moves_percentage"] = 0

activities_moves = dict(activities_moves)

alignments_metrics = {
    "activities_moves": activities_moves,
}

# calculate total number of perfectly fitting traces
perfect_traces = sum(1 for res in aligned_traces if res.get("fitness", 0.0) >= 1.0)

conformance_statistics = {
    "perfect_fitting_traces": perfect_traces,
    "total_traces": len(aligned_traces)
}

print(f"📊 Conformance statistics: {perfect_traces}/{len(aligned_traces)} perfect traces ({(perfect_traces/len(aligned_traces)*100):.1f}%)")

📊 Computing model moves, log moves, and percentages per activity...
📊 Conformance statistics: 2/123 perfect traces (1.6%)


### Helper to create deviation view

- since we only have the flows we dont know where to place the activities
- because of that we need to know the activity predecessors

In [ ]:
deviation_predecessors = {}

for activity in activities_deviations.keys():
    df_sorted = df_pm.sort_values(["case:concept:name", "time:timestamp"])
    df_sorted['prev_activity'] = df_sorted.groupby("case:concept:name")["concept:name"].shift(1)
    occurrences = df_sorted[df_sorted["concept:name"] == activity].copy()
    occurrences['prev_activity'] = occurrences['prev_activity'].fillna("<<start>>")
    # remove occurrences where the predecessor is the same as the activity itself
    occurrences = occurrences[occurrences['prev_activity'] != activity]
    predecessor_counts = (
        occurrences['prev_activity']
        .value_counts()
        .reset_index()
    )
    predecessor_counts.columns = ['predecessor', 'frequency']
    deviation_predecessors[activity] = predecessor_counts.to_dict(orient='records')

deviation_successors = {}

for activity in activities_deviations.keys():
    df_sorted = df_pm.sort_values(["case:concept:name", "time:timestamp"])
    df_sorted['next_activity'] = df_sorted.groupby("case:concept:name")["concept:name"].shift(-1)
    occurrences = df_sorted[df_sorted["concept:name"] == activity].copy()
    occurrences['next_activity'] = occurrences['next_activity'].fillna("<<end>>")
    # remove occurrences where the successor is the same as the activity itself
    occurrences = occurrences[occurrences['next_activity'] != activity]
    successor_counts = (
        occurrences['next_activity']
        .value_counts()
        .reset_index()
    )
    successor_counts.columns = ['successor', 'frequency']
    deviation_successors[activity] = successor_counts.to_dict(orient='records')


### Structure last variables

In [ ]:
conformance_metrics = {
    "time_conformance": time_conformance,
    "conformance_profile": conformance_profile,
    "conformance_statistics": conformance_statistics,
    "alignments_metrics": alignments_metrics
}

helper_variables = {
    "model_next_activities": model_next_activities,
    "roles_lanes": roles_lanes,
    "activities_roles": activities_roles,
    "deviation_predecessors": deviation_predecessors,
    "deviation_successors": deviation_successors
}

### [OPTIONAL] Evaluation Controlled Prototype - Print Metrics Report

When `EXPORT_EVALUATION_METRICS=true` in `.env`, the next cell writes a controlled-prototype Markdown report under `evaluation/controlled-prototype/results`. The report mirrors the main frontend views: `Performance`, `Alignments`, and `Frequency`. It keeps the same metadata style used by the activity-mapping validation report and records the metric values before frontend BPMN augmentation, so the exported tables can be compared with the rendered frontend views.


In [ ]:
# optional controlled-prototype evaluation metrics export.
# runs silently when EXPORT_EVALUATION_METRICS=false.
from pathlib import Path
from datetime import datetime
import hashlib
import importlib.metadata
import math
import platform
import subprocess
import sys


def _report_file_sha256(path):
    path = Path(path)
    if not path.exists() or not path.is_file():
        return None
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _report_package_version(package_name):
    try:
        return importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        return None


def _report_command_output(args):
    try:
        return subprocess.check_output(args, text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        return None


def _report_repository_root():
    root = _report_command_output(["git", "rev-parse", "--show-toplevel"])
    return Path(root) if root else Path.cwd()


def _report_repo_relative_path(path):
    if path is None:
        return None
    path = Path(path)
    try:
        return path.resolve().relative_to(_report_repository_root().resolve()).as_posix()
    except Exception:
        return str(path)


def _report_markdown_escape(value):
    if value is None:
        return "N/A"
    text = str(value)
    return text.replace("|", "\\|").replace("\n", "<br>")


def _report_format_number(value):
    if value is None:
        return "N/A"
    if isinstance(value, bool):
        return "Yes" if value else "No"
    if isinstance(value, float):
        return f"{value:.4f}"
    return value


def _report_format_duration(seconds):
    if seconds is None:
        return "N/A"
    try:
        if pandas.isna(seconds):
            return "N/A"
    except Exception:
        pass
    try:
        seconds = float(seconds)
    except (TypeError, ValueError):
        return "N/A"
    if not math.isfinite(seconds):
        return "N/A"
    if seconds < 0:
        return "0s"
    total_seconds = int(math.floor(seconds))
    days = total_seconds // 86400
    hours = (total_seconds % 86400) // 3600
    minutes = (total_seconds % 3600) // 60
    secs = total_seconds % 60
    parts = []
    if days > 0:
        parts.append(f"{days}d")
    if hours > 0:
        parts.append(f"{hours}h")
    if minutes > 0:
        parts.append(f"{minutes}m")
    if secs > 0 or not parts:
        parts.append(f"{secs}s")
    return " ".join(parts)


def _report_markdown_table(headers, rows):
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join("---" for _ in headers) + " |",
    ]
    if not rows:
        lines.append("| " + " | ".join(["No data"] + ["" for _ in headers[1:]]) + " |")
        return "\n".join(lines)
    for row in rows:
        lines.append("| " + " | ".join(_report_markdown_escape(_report_format_number(value)) for value in row) + " |")
    return "\n".join(lines)


def _report_display_activity(label, info=None):
    if label in {"<<start>>", "<<end>>"}:
        return label
    if isinstance(info, dict) and info.get("originalActivity"):
        return info["originalActivity"]
    return normalized_to_original.get(label, label) if "normalized_to_original" in globals() else label


def _report_display_flow_activity(label):
    if label in {"<<start>>", "<<end>>"}:
        return label
    return normalized_to_original.get(label, label) if "normalized_to_original" in globals() else label


def _report_bool_text(value):
    if value is None:
        return "N/A"
    return "Yes" if bool(value) else "No"


def _report_runtime_weight_discrepancies(weights):
    expected = {
        "action": 0.30,
        "object": 0.20,
        "full_label": 0.25,
        "lexical": 0.10,
        "token": 0.15,
    }
    discrepancies = []
    for key, expected_value in expected.items():
        runtime_value = weights.get(key)
        if runtime_value is None or abs(runtime_value - expected_value) > 1e-9:
            discrepancies.append(f"{key}: expected {expected_value:.4f}, runtime {_report_format_number(runtime_value)}")
    if abs(CANDIDATE_THRESHOLD - 0.80) > 1e-9:
        discrepancies.append(f"candidate_threshold: expected 0.8000, runtime {CANDIDATE_THRESHOLD:.4f}")
    if abs(MINIMUM_CANDIDATE_MARGIN - 0.05) > 1e-9:
        discrepancies.append(f"minimum_candidate_margin: expected 0.0500, runtime {MINIMUM_CANDIDATE_MARGIN:.4f}")
    return discrepancies


def _report_semantic_penalty_description(config):
    if not isinstance(config, dict):
        return "N/A"
    return (
        f"0.00 when no usable verb is identified; 0.00 when extracted verb lemmas agree; "
        f"0.00 when action similarity >= {config['high_similarity_floor']:.2f}; "
        f"{config['medium_similarity_penalty']:.2f} when action similarity >= {config['medium_similarity_floor']:.2f} and < {config['high_similarity_floor']:.2f}; "
        f"{config['low_similarity_penalty']:.2f} when action similarity >= {config['low_similarity_floor']:.2f} and < {config['medium_similarity_floor']:.2f}; "
        f"{config['very_low_similarity_penalty']:.2f} when action similarity < {config['low_similarity_floor']:.2f}"
    )


def _report_activity_rows(activity_metrics, metric_kind):
    rows = []
    if not isinstance(activity_metrics, dict):
        return rows
    for activity, info in sorted(activity_metrics.items(), key=lambda item: _report_display_activity(item[0], item[1])):
        if not isinstance(info, dict):
            continue
        activity_name = _report_display_activity(activity, info)
        in_model = info.get("inModel", activity in model_activities if "model_activities" in globals() else None)
        if metric_kind == "performance":
            rows.append([
                activity_name,
                _report_bool_text(in_model),
                _report_format_duration(info.get("avg_sojourn")),
                _report_format_duration(info.get("median_sojourn")),
                _report_format_duration(info.get("min_sojourn")),
                _report_format_duration(info.get("max_sojourn")),
                info.get("event_count"),
            ])
        elif metric_kind == "frequency":
            rows.append([
                activity_name,
                _report_bool_text(in_model),
                info.get("frequency"),
                info.get("mismatch_incoming", "N/A"),
                info.get("mismatch_outgoing", "N/A"),
            ])
    return rows


def _report_flow_rows(flow_metrics, metric_kind, flow_type):
    rows = []
    if not isinstance(flow_metrics, list):
        return rows
    for flow in sorted(flow_metrics, key=lambda item: (_report_display_flow_activity(item.get("from")), _report_display_flow_activity(item.get("to")))):
        if metric_kind == "performance":
            rows.append([
                _report_display_flow_activity(flow.get("from")),
                _report_display_flow_activity(flow.get("to")),
                _report_format_duration(flow.get("avg_sojourn")),
                _report_format_duration(flow.get("median_sojourn")),
                flow_type,
            ])
        elif metric_kind == "frequency":
            rows.append([
                _report_display_flow_activity(flow.get("from")),
                _report_display_flow_activity(flow.get("to")),
                flow.get("frequency"),
                flow_type,
            ])
    return rows


def _report_alignment_rows(activities_moves):
    rows = []
    if not isinstance(activities_moves, dict):
        return rows
    for activity, moves in sorted(activities_moves.items(), key=lambda item: _report_display_activity(item[0], item[1])):
        if not isinstance(moves, dict):
            continue
        rows.append([
            _report_display_activity(activity, moves),
            moves.get("synchronous_moves", 0),
            moves.get("log_moves", 0),
            moves.get("model_moves", 0),
            moves.get("synchronous_moves_percentage", 0),
            moves.get("log_moves_percentage", 0),
            moves.get("model_moves_percentage", 0),
        ])
    return rows


def _report_metadata(run_id, created_at, notebook_path, env_path):
    linguistic_pipeline_version = None
    try:
        linguistic_pipeline_version = _report_package_version(LINGUISTIC_PIPELINE.replace("_", "-"))
    except Exception:
        linguistic_pipeline_version = None

    weights = {
        "action": ACTION_WEIGHT,
        "object": OBJECT_WEIGHT,
        "full_label": FULL_LABEL_WEIGHT,
        "lexical": LEXICAL_WEIGHT,
        "token": TOKEN_WEIGHT,
    }
    return {
        "run_id": run_id,
        "created_at": created_at,
        "git_commit": _report_command_output(["git", "rev-parse", "HEAD"]),
        "notebook_path": _report_repo_relative_path(notebook_path),
        "notebook_sha256": _report_file_sha256(notebook_path),
        "env_path": _report_repo_relative_path(env_path),
        "env_sha256": _report_file_sha256(env_path),
        "operating_system": platform.platform(),
        "python_version": sys.version.replace("\n", " "),
        "sentence_transformers_version": _report_package_version("sentence-transformers"),
        "embedding_model": EMBEDDING_MODEL,
        "embedding_model_revision": "N/A",
        "spacy_version": _report_package_version("spacy"),
        "linguistic_pipeline": LINGUISTIC_PIPELINE,
        "linguistic_pipeline_version": linguistic_pipeline_version,
        "rapidfuzz_version": _report_package_version("rapidfuzz"),
        "execution_device": str(getattr(globals().get("model", None), "device", "N/A")),
        "weights": weights,
        "candidate_threshold": CANDIDATE_THRESHOLD,
        "minimum_candidate_margin": MINIMUM_CANDIDATE_MARGIN,
        "configuration_discrepancies": _report_runtime_weight_discrepancies(weights),
        "penalties": _report_semantic_penalty_description(globals().get("SEMANTIC_VERB_PENALTY_CONFIG")),
        "export_evaluation_metrics": EXPORT_EVALUATION_METRICS,
    }


if EXPORT_EVALUATION_METRICS:
    run_id = datetime.now().strftime("%Y-%m-%d-%H%M%S")
    created_at = datetime.now().astimezone().isoformat(timespec="seconds")
    controlled_prototype_dir = Path("evaluation") / "controlled-prototype"
    results_dir = controlled_prototype_dir / "results"
    results_dir.mkdir(parents=True, exist_ok=True)

    event_log_path = Path(file_path) if "file_path" in globals() and file_path else None
    bpmn_model_path = Path(bpmn_path) if "bpmn_path" in globals() and bpmn_path else None
    notebook_path = Path("pipeline.ipynb")
    env_path = Path(".env")
    markdown_output_path = results_dir / f"{run_id}.md"

    metadata = _report_metadata(run_id, created_at, notebook_path, env_path)

    discrepancy_text = "; ".join(metadata["configuration_discrepancies"]) if metadata["configuration_discrepancies"] else "None"

    lines = [
        "# Controlled Prototype Evaluation Metrics Report",
        "",
        "## 1. Run Metadata",
        "",
        _report_markdown_table(
            ["Field", "Value"],
            [
                ["Run identifier", metadata["run_id"]],
                ["Timestamp", metadata["created_at"]],
                ["Prototype Git commit or release", metadata["git_commit"]],
                ["Evaluation notebook", metadata["notebook_path"]],
                ["Evaluation notebook SHA-256", metadata["notebook_sha256"]],
                ["Validation .env", metadata["env_path"]],
                ["Validation .env SHA-256", metadata["env_sha256"]],
                ["Operating system", metadata["operating_system"]],
                ["Python version", metadata["python_version"]],
                ["sentence-transformers version", metadata["sentence_transformers_version"]],
                ["Embedding model", metadata["embedding_model"]],
                ["Embedding model revision", metadata["embedding_model_revision"]],
                ["spaCy version", metadata["spacy_version"]],
                ["Linguistic pipeline", metadata["linguistic_pipeline"]],
                ["Linguistic pipeline version", metadata["linguistic_pipeline_version"]],
                ["RapidFuzz version", metadata["rapidfuzz_version"]],
                ["CPU/GPU execution device", metadata["execution_device"]],
                ["Action weight", metadata["weights"]["action"]],
                ["Object weight", metadata["weights"]["object"]],
                ["Full-label semantic weight", metadata["weights"]["full_label"]],
                ["Lexical weight", metadata["weights"]["lexical"]],
                ["Token weight", metadata["weights"]["token"]],
                ["Candidate threshold", metadata["candidate_threshold"]],
                ["Minimum candidate margin", metadata["minimum_candidate_margin"]],
                ["Semantic verb penalty", metadata["penalties"]],
                ["Configuration discrepancy flag", discrepancy_text],
                ["Export evaluation metrics", metadata["export_evaluation_metrics"]],
            ],
        ),
        "",
        "### Input Hashes",
        "",
        _report_markdown_table(
            ["Artifact", "Path", "SHA-256"],
            [
                ["BPMN", _report_repo_relative_path(bpmn_model_path), _report_file_sha256(bpmn_model_path)],
                ["Event log", _report_repo_relative_path(event_log_path), _report_file_sha256(event_log_path)],
            ],
        ),
        "",
        "## 2. Performance View",
        "",
    ]

    performance_sections = performance_metrics if "performance_metrics" in globals() and isinstance(performance_metrics, dict) else {}
    for section_key, section_title in (("all", "All Activities"), ("model", "Model Activities"), ("deviations", "Deviation Activities")):
        lines.extend([
            f"### {section_title}",
            "",
            _report_markdown_table(
                ["Activity", "In model", "Mean duration", "Median duration", "Min. duration", "Max. duration", "Event count"],
                _report_activity_rows(performance_sections.get("activities", {}).get(section_key, {}), "performance"),
            ),
            "",
        ])
    for section_key, section_title in (("all", "All Flows"), ("model", "Model Flows"), ("deviations", "Deviation Flows")):
        lines.extend([
            f"### {section_title}",
            "",
            _report_markdown_table(
                ["From", "To", "Mean waiting time", "Median waiting time", "Type"],
                _report_flow_rows(performance_sections.get("flows", {}).get(section_key, []), "performance", section_key),
            ),
            "",
        ])

    alignments_sections = conformance_metrics.get("alignments_metrics", {}) if "conformance_metrics" in globals() and isinstance(conformance_metrics, dict) else {}
    conformance_statistics_section = conformance_metrics.get("conformance_statistics", {}) if "conformance_metrics" in globals() and isinstance(conformance_metrics, dict) else {}
    conformance_profile_section = conformance_metrics.get("conformance_profile", {}) if "conformance_metrics" in globals() and isinstance(conformance_metrics, dict) else {}
    lines.extend([
        "## 3. Alignments View",
        "",
        "### Activity Moves",
        "",
        _report_markdown_table(
            ["Activity", "Synchronous moves", "Log moves", "Model moves", "Sync %", "Log %", "Model %"],
            _report_alignment_rows(alignments_sections.get("activities_moves", {})),
        ),
        "",
        "### Conformance Statistics",
        "",
        _report_markdown_table(
            ["Metric", "Value"],
            [[key, value] for key, value in sorted(conformance_statistics_section.items())],
        ),
        "",
        "### Conformance Profile",
        "",
        _report_markdown_table(
            ["Metric", "Value"],
            [[key, value] for key, value in sorted(conformance_profile_section.items())],
        ),
        "",
        "## 4. Frequency View",
        "",
    ])

    frequency_sections = frequency_metrics if "frequency_metrics" in globals() and isinstance(frequency_metrics, dict) else {}
    for section_key, section_title in (("all", "All Activities"), ("model", "Model Activities"), ("deviations", "Deviation Activities")):
        lines.extend([
            f"### {section_title}",
            "",
            _report_markdown_table(
                ["Activity", "In model", "Frequency", "Incoming mismatch", "Outgoing mismatch"],
                _report_activity_rows(frequency_sections.get("activities", {}).get(section_key, {}), "frequency"),
            ),
            "",
        ])
    for section_key, section_title in (("all", "All Flows"), ("model", "Model Flows"), ("deviations", "Deviation Flows")):
        lines.extend([
            f"### {section_title}",
            "",
            _report_markdown_table(
                ["From", "To", "Frequency", "Type"],
                _report_flow_rows(frequency_sections.get("flows", {}).get(section_key, []), "frequency", section_key),
            ),
            "",
        ])

    markdown_output_path.write_text("\n".join(lines), encoding="utf-8")


### Analysis Complete - Results Ready

The conformance analysis pipeline has been completed successfully. All analysis variables are now available for use by the backend API.

In [ ]:
# store analysis results for API access (silent execution)
# this cell prepares all analysis results without displaying them

# convert activity to frontend keys
def _frontend_activity_label(label):
    if label in ("<<start>>", "START_OF_CASE"):
        return "<<start>>"
    if label in ("<<end>>", "END_OF_CASE"):
        return "<<end>>"

    frontend_label = normalize_frontend_key(label)
    if frontend_label == "start-of-case":
        return "<<start>>"
    if frontend_label == "end-of-case":
        return "<<end>>"
    return frontend_label

def _frontend_activity_metrics(activity_metrics):
    if not isinstance(activity_metrics, dict):
        return activity_metrics

    frontend_metrics = {}
    for activity_key, activity_info in activity_metrics.items():
        original_activity = activity_key
        if isinstance(activity_info, dict):
            original_activity = activity_info.get("originalActivity", activity_key)
        frontend_metrics[_frontend_activity_label(original_activity)] = activity_info
    return frontend_metrics

def _frontend_activity_keyed_dict(activity_dict):
    if not isinstance(activity_dict, dict):
        return activity_dict
    return {
        _frontend_activity_label(activity_key): activity_value
        for activity_key, activity_value in activity_dict.items()
    }

def _frontend_activity_list(activity_list):
    if not isinstance(activity_list, list):
        return activity_list
    return [
        _frontend_activity_label(activity)
        if isinstance(activity, str) else activity
        for activity in activity_list
    ]

def _frontend_flow_items(flow_items):
    if not isinstance(flow_items, list):
        return flow_items

    activity_fields = {"from", "to", "source", "target", "activity", "predecessor", "successor"}
    frontend_items = []
    for item in flow_items:
        if not isinstance(item, dict):
            frontend_items.append(item)
            continue
        frontend_item = dict(item)
        for field in activity_fields:
            if isinstance(frontend_item.get(field), str):
                frontend_item[field] = _frontend_activity_label(frontend_item[field])
        frontend_items.append(frontend_item)
    return frontend_items

def _frontend_sectioned_activity_dict(sectioned_dict):
    if not isinstance(sectioned_dict, dict):
        return sectioned_dict
    return {
        section_name: _frontend_activity_keyed_dict(section_values)
        for section_name, section_values in sectioned_dict.items()
    }

def _frontend_sectioned_flows(sectioned_flows):
    if not isinstance(sectioned_flows, dict):
        return sectioned_flows
    return {
        section_name: _frontend_flow_items(section_values)
        for section_name, section_values in sectioned_flows.items()
    }

# overwrite the original metrics with frontend-friendly keys and structures

if "performance_metrics" in locals() and isinstance(performance_metrics, dict):
    performance_metrics["activities"] = {
        section_name: _frontend_activity_metrics(section_metrics)
        for section_name, section_metrics in performance_metrics.get("activities", {}).items()
    }
    performance_metrics["flows"] = _frontend_sectioned_flows(performance_metrics.get("flows", {}))

if "frequency_metrics" in locals() and isinstance(frequency_metrics, dict):
    frequency_metrics["activities"] = _frontend_sectioned_activity_dict(frequency_metrics.get("activities", {}))
    frequency_metrics["flows"] = _frontend_sectioned_flows(frequency_metrics.get("flows", {}))

if "alignments_metrics" in locals() and isinstance(alignments_metrics, dict):
    alignments_metrics["activities_moves"] = _frontend_activity_keyed_dict(
        alignments_metrics.get("activities_moves", {})
    )

if "conformance_metrics" in locals() and isinstance(conformance_metrics, dict):
    conformance_metrics["time_conformance"] = _frontend_sectioned_flows(
        conformance_metrics.get("time_conformance", {})
    )
    if isinstance(conformance_metrics.get("alignments_metrics"), dict):
        conformance_metrics["alignments_metrics"]["activities_moves"] = _frontend_activity_keyed_dict(
            conformance_metrics["alignments_metrics"].get("activities_moves", {})
        )

if "helper_variables" in locals() and isinstance(helper_variables, dict):
    helper_variables["model_next_activities"] = {
        _frontend_activity_label(activity): _frontend_activity_list(next_activities)
        for activity, next_activities in helper_variables.get("model_next_activities", {}).items()
    }
    if isinstance(helper_variables.get("activities_roles"), dict):
        helper_variables["activities_roles"] = _frontend_sectioned_activity_dict(
            helper_variables.get("activities_roles", {})
        )
    helper_variables["deviation_predecessors"] = {
        _frontend_activity_label(activity): _frontend_flow_items(predecessors)
        for activity, predecessors in helper_variables.get("deviation_predecessors", {}).items()
    }
    helper_variables["deviation_successors"] = {
        _frontend_activity_label(activity): _frontend_flow_items(successors)
        for activity, successors in helper_variables.get("deviation_successors", {}).items()
    }

# variables to expose for the API
variables_to_extract = [
    'performance_metrics',
    'frequency_metrics',
    'alignments_metrics',
    'conformance_metrics',
    'helper_variables'
]

# trigger lazy evaluation for sized containers to ensure they are fully computed before API access
for _var in variables_to_extract:
    if _var in locals():
        try:
            # trigger potential lazy evaluation (safe for sized containers)
            _ = len(locals()[_var])
        except Exception:
            # if len() is not supported or evaluation fails, ignore silently
            pass

# ensure other key variables are accessible
analysis_complete = True
print("✅ Analysis results prepared for API access")